# 🔐 Secure Login System with Password Hashing
**IBM SkillsBuild Cyber Security 2026 — BharatCares**

| Field | Details |
|---|---|
| **Student Name** | StudentName |
| **Project Name** | Secure Login System with Password Hashing |
| **Domain** | Web Security / Authentication |
| **Tech Stack** | Python, Flask, SQLite, bcrypt |
| **Program** | IBM SkillsBuild Cyber Security 2026 |

---

## 📌 Project Overview
This project demonstrates a **secure user authentication system** built with Python and Flask.  
It covers five core cybersecurity concepts:
1. **Password Hashing** — passwords are never stored in plain text
2. **Salting** — bcrypt automatically generates a unique salt per password
3. **SQL Injection Prevention** — all database queries use parameterized statements
4. **Brute-force Lockout** — accounts lock after 5 consecutive failed login attempts
5. **Session Management** — server-side sessions protect access to restricted pages

---
## Section 1 — Install Dependencies

In [ ]:
# Run this cell once to install all required libraries
import sys
!{sys.executable} -m pip install flask flask-bcrypt bcrypt

---
## Section 2 — Database Setup (`database.py`)

Creates a local SQLite database with a `users` table.  
Fields: `id`, `username`, `password` (hashed), `failed_attempts`, `is_locked`.

In [ ]:
import sqlite3

def init_db():
    """Initialize the SQLite database and create the users table."""
    conn = sqlite3.connect("users.db")
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id               INTEGER PRIMARY KEY AUTOINCREMENT,
            username         TEXT UNIQUE NOT NULL,
            password         TEXT NOT NULL,
            failed_attempts  INTEGER DEFAULT 0,
            is_locked        INTEGER DEFAULT 0
        )
    """)
    conn.commit()
    conn.close()
    print("[✔] Database initialized successfully.")

# Initialize the database
init_db()

---
## Section 3 — Security Concept Demo: Password Hashing & Salting

**Why hashing?** If a database is stolen, attackers cannot read plain-text passwords.  
**Why salting?** Even if two users have the same password, their hashes will be different.

In [ ]:
from flask_bcrypt import Bcrypt
from flask import Flask

app = Flask(__name__)
bcrypt = Bcrypt(app)

# --- Demonstrate password hashing ---
plain_password = "mypassword123"

hash1 = bcrypt.generate_password_hash(plain_password).decode("utf-8")
hash2 = bcrypt.generate_password_hash(plain_password).decode("utf-8")

print("Plain text password :", plain_password)
print()
print("Hash #1 (bcrypt)     :", hash1)
print("Hash #2 (bcrypt)     :", hash2)
print()
print("Are the two hashes identical?", hash1 == hash2)  # False — different salts!
print()
print("Does hash1 match the password?", bcrypt.check_password_hash(hash1, plain_password))  # True
print("Does hash2 match wrong password?", bcrypt.check_password_hash(hash2, "wrongpass"))   # False

---
## Section 4 — Security Concept Demo: SQL Injection Prevention

**SQL Injection** is one of the most common web attacks (OWASP Top 10).  
We prevent it by using **parameterized queries** — never string concatenation.

In [ ]:
import sqlite3

conn = sqlite3.connect("users.db")

# ❌ VULNERABLE — DO NOT DO THIS
# username = "admin' OR '1'='1"  # classic SQL injection payload
# query = f"SELECT * FROM users WHERE username = '{username}'"  # DANGEROUS!
# This would return ALL users because '1'='1' is always true.

# ✅ SAFE — Parameterized query (what our app uses)
username = "admin' OR '1'='1"  # attacker's payload
cursor = conn.execute(
    "SELECT * FROM users WHERE username = ?",  # '?' is a safe placeholder
    (username,)                                # value is passed separately
)
result = cursor.fetchone()
conn.close()

print("SQL Injection attempt result with parameterized query:", result)
print("✔ Attack neutralized — no rows returned despite malicious input.")

---
## Section 5 — Full Application Code (`app.py`)

The complete Flask web application with all routes and security features.

In [ ]:
from flask import Flask, render_template, request, redirect, session, url_for
from flask_bcrypt import Bcrypt
import sqlite3

app = Flask(__name__)
app.secret_key = "cyber_secret_key_2024"
bcrypt = Bcrypt(app)

MAX_ATTEMPTS = 5  # Lock account after this many wrong passwords


def get_db():
    """Return a new database connection."""
    return sqlite3.connect("users.db")


# ── REGISTER ──────────────────────────────────────────────────────────────────
@app.route("/register", methods=["GET", "POST"])
def register():
    error = None
    if request.method == "POST":
        username = request.form["username"].strip()
        password = request.form["password"]

        if len(password) < 6:
            error = "Password must be at least 6 characters."
        else:
            # Hash the password with bcrypt (auto-salted, one-way)
            hashed_pw = bcrypt.generate_password_hash(password).decode("utf-8")
            try:
                conn = get_db()
                conn.execute(
                    "INSERT INTO users (username, password) VALUES (?, ?)",
                    (username, hashed_pw)  # parameterized — safe from SQL injection
                )
                conn.commit()
                conn.close()
                return redirect(url_for("login"))
            except sqlite3.IntegrityError:
                error = "Username already exists. Please choose another."

    return render_template("register.html", error=error)


# ── LOGIN ─────────────────────────────────────────────────────────────────────
@app.route("/login", methods=["GET", "POST"])
def login():
    error = None
    if request.method == "POST":
        username = request.form["username"].strip()
        password = request.form["password"]

        conn = get_db()
        user = conn.execute(
            "SELECT * FROM users WHERE username = ?", (username,)
        ).fetchone()
        conn.close()

        if user is None:
            error = "Invalid username or password."
        elif user[4] == 1:  # is_locked
            error = "Account locked after too many failed attempts."
        elif bcrypt.check_password_hash(user[2], password):
            # Correct — reset counter, start session
            conn = get_db()
            conn.execute("UPDATE users SET failed_attempts = 0 WHERE username = ?", (username,))
            conn.commit()
            conn.close()
            session["user"] = username
            return redirect(url_for("dashboard"))
        else:
            # Wrong password — increment attempt counter
            conn = get_db()
            new_attempts = user[3] + 1
            locked = 1 if new_attempts >= MAX_ATTEMPTS else 0
            conn.execute(
                "UPDATE users SET failed_attempts = ?, is_locked = ? WHERE username = ?",
                (new_attempts, locked, username)
            )
            conn.commit()
            conn.close()
            if locked:
                error = "Account locked after too many failed attempts."
            else:
                remaining = MAX_ATTEMPTS - new_attempts
                error = f"Wrong password. {remaining} attempt(s) remaining before lockout."

    return render_template("login.html", error=error)


# ── DASHBOARD (protected) ─────────────────────────────────────────────────────
@app.route("/dashboard")
def dashboard():
    if "user" not in session:
        return redirect(url_for("login"))  # redirect unauthenticated users
    return render_template("dashboard.html", username=session["user"])


# ── LOGOUT ────────────────────────────────────────────────────────────────────
@app.route("/logout")
def logout():
    session.pop("user", None)
    return redirect(url_for("login"))


# ── HOME REDIRECT ─────────────────────────────────────────────────────────────
@app.route("/")
def home():
    return redirect(url_for("login"))


print("[✔] Flask app defined with routes: /, /register, /login, /dashboard, /logout")

---
## Section 6 — Run the Application

> ⚠️ **Run this cell last.** Once running, open your browser at `http://127.0.0.1:5000`  
> To stop the server, press **Interrupt Kernel** (■ button in Jupyter toolbar) or `Ctrl+C` in terminal.

In [ ]:
if __name__ == "__main__":
    init_db()
    print("[✔] Starting Secure Login App — open http://127.0.0.1:5000 in your browser")
    app.run(debug=True, use_reloader=False)

---
## Section 7 — Unit Tests

Automated tests to verify core security behaviour without a browser.

In [ ]:
import os

# Use a separate test database so tests don't affect real data
TEST_DB = "test_users.db"

def init_test_db():
    conn = sqlite3.connect(TEST_DB)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            password TEXT NOT NULL,
            failed_attempts INTEGER DEFAULT 0,
            is_locked INTEGER DEFAULT 0
        )
    """)
    conn.commit()
    conn.close()

def teardown_test_db():
    if os.path.exists(TEST_DB):
        os.remove(TEST_DB)

# ── Test 1: Password hash is not plain text ───────────────────────────────────
def test_password_not_plain_text():
    with app.app_context():
        pw = "secret123"
        hashed = bcrypt.generate_password_hash(pw).decode("utf-8")
        assert hashed != pw, "FAIL: Password stored as plain text!"
        assert hashed.startswith("$2b$"), "FAIL: Not a bcrypt hash!"
    print("✔ Test 1 PASSED — password is not stored as plain text")

# ── Test 2: Same password produces different hashes (salting) ─────────────────
def test_salting():
    with app.app_context():
        pw = "secret123"
        h1 = bcrypt.generate_password_hash(pw).decode("utf-8")
        h2 = bcrypt.generate_password_hash(pw).decode("utf-8")
        assert h1 != h2, "FAIL: Same hash produced — no salting!"
    print("✔ Test 2 PASSED — same password produces different hashes (salting works)")

# ── Test 3: Register page returns 200 ─────────────────────────────────────────
def test_register_page_loads():
    with app.test_client() as client:
        response = client.get("/register")
        assert response.status_code == 200, "FAIL: Register page did not load!"
    print("✔ Test 3 PASSED — register page loads (200 OK)")

# ── Test 4: Login page returns 200 ────────────────────────────────────────────
def test_login_page_loads():
    with app.test_client() as client:
        response = client.get("/login")
        assert response.status_code == 200, "FAIL: Login page did not load!"
    print("✔ Test 4 PASSED — login page loads (200 OK)")

# ── Test 5: Dashboard blocks unauthenticated access ───────────────────────────
def test_dashboard_requires_login():
    with app.test_client() as client:
        response = client.get("/dashboard")
        assert response.status_code == 302, "FAIL: Dashboard should redirect!"
        assert "/login" in response.location, "FAIL: Should redirect to login!"
    print("✔ Test 5 PASSED — dashboard blocks unauthenticated users")

# ── Run all tests ─────────────────────────────────────────────────────────────
print("Running security unit tests...\n")
test_password_not_plain_text()
test_salting()
test_register_page_loads()
test_login_page_loads()
test_dashboard_requires_login()
print("\n✅ All tests passed!")

---
## Section 8 — Summary & Conclusions

### What Was Built
A fully functional, secure web-based login system using Python and Flask that demonstrates five key cybersecurity concepts in a real application.

### Security Concepts Demonstrated

| Concept | Implementation | Importance |
|---|---|---|
| Password Hashing | bcrypt one-way hash | Protects passwords if database is breached |
| Salting | bcrypt auto-salt per hash | Prevents rainbow table attacks |
| SQL Injection Prevention | Parameterized queries | Prevents data theft / manipulation |
| Brute-force Lockout | 5-attempt account lock | Prevents automated password guessing |
| Session Management | Flask server-side sessions | Restricts access to authenticated users only |

### Key Takeaways
- **Never store passwords in plain text.** Always hash with a strong algorithm like bcrypt.
- **Never build SQL queries by string concatenation.** Always use parameterized queries.
- **Rate-limit login attempts** to defend against brute-force attacks.
- **Sessions must be server-side** — never trust client-side storage for authentication state.

### References
- OWASP Top 10: https://owasp.org/www-project-top-ten/
- Flask Documentation: https://flask.palletsprojects.com/
- bcrypt Algorithm: https://en.wikipedia.org/wiki/Bcrypt
- NIST Password Guidelines: https://pages.nist.gov/800-63-3/sp800-63b.html